In python function can be defined insdie a function

In [ ]:
def outer():
    x = 10 
    
    def inner():
        print(x)
    inner()

outer()

# Inner can access x even though it is outside of the function

10


LEGB - Rules \
L - Local  \
E - Enclosing  \
G - Global  \
B - Build-In 

Order follow this / 
Initaially search local  if they don't fine goes for enclosing, then global andd then build in

In [ ]:
def outer():
    x = 20 
    print("Outer")
    
    def inner():
        print("Inner")
        print(x)
        
    return inner
    # retrun inner()  => It returns
fn = outer()
fn()

#  print(outer()()) 2 times and prints none
        

Outer
Inner
20


In [ ]:
def outer():
    x = 10
    
    def inner():
        print(x)
        
    x = 50 
    
    return inner

fn = outer()
fn()

50


In [17]:
def outer():
    count = 0
    
    def increment():
        count = count + 1
        return count
    return increment

counter = outer()
print(counter())


UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

In [ ]:
def outer():
    count = 0
    print(count)
    
    def increment():
        nonlocal count
        count = count + 1
        return count
    return increment

counter = outer()
print(counter())
print(counter())
print(counter())


0
1
2
3


Case 1: only reading, no assignment 
Case 2: reading and assigning, without nonlocal we get #UnboundLocalError

"Idiomatic" in the way experienced user of a lnaguage naturally write it, using the language's own convetions 
and features rather than pattern borrowed from elsewhere.

In [ ]:
# Not idiomtic looks like Java or C style 
items = ["a","b","c"]
i = 0
while i < len(items):
    print(i, items[i])
    i = i+1
    
# This is call idiomtic
for index,item in enumerate(items):
    print(index, item)


0 a
1 b
2 c
0 a
1 b
2 c


nonlocal count tells python to rebind count to nearest enclsoing function scope where it exists 
Unlike global, it doesn't target module level variable

In [ ]:
def outer():
    count = 1
    
    def increment():
        # print(count) # It creates an error
        count = 100  # We are assinging first that's why it is working
        #count = count + 1
        return count
    
    return increment
        
fn = outer()
print(fn())

# print alone doesn’t change anything. It only reads, so outer’s count stays the same.
# One assignment anywhere in increment changes the picture. Even a line you never expect 
# to run, like count = 5 at the bottom or inside an if False: block, makes count local for
# the whole function, and the print(count) above it would then raise UnboundLocalError.

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

Decorators 

A decorator is callable that takes another callable (typically fucntion) and retruns a 
replacement callable, often adding a behavioud

In [55]:
def greet():
    print("Hello")

def decorator(func):
    def wrapper():
        print("Before")
        func()
        print("After")
    return  wrapper

#  decorator(greet)()  one way is this 

output = decorator(greet) # Decorator is taking another function as input
output()
print("___________________________________")

output2 = decorator(print("Why"))
output2()


Before
Hello
After
___________________________________
Why
Before


TypeError: 'NoneType' object is not callable

1. output2 = decorator(print("why"))  it prints why and gives print(why) gives None 
2. Before executes 
3. func reaplces this function whcih is None so we get typeError

In [59]:
def greet2():
    print("Hello")

def decorator2(func):
    print("Before")
    func()
    print("After")
    return decorator2

greet = decorator2(greet2)
greet()

Before
Hello
After


TypeError: decorator2() missing 1 required positional argument: 'func'

We forgot about the wrapper function

In [ ]:
# A Closure is a function that remembers the vaiables from the scope where it was created, even after that
# outer scope has finished running

def make_multiple(x):
    def multiply(n):
        return x * n
    return multiply

double = make_multiple(2)
triple = make_multiple(3)

print(double(5))
print(triple(5))

# Make mutiple is already completed yet inner method multiply still has access to X. The captured variable is a closure

# Three conditions 
# 1. There's is a nested fucntion 
# 2. The inner function uses a varaible from the enclosing fucntion 
# 3. The enclsoing function retrun the inner function


10
15


In [ ]:
def something(func):
    print("A")
    
    def wrapper():
        print("B")
        func()
        print("C")
        
    print("D")
    return wrapper

@something
def greet():
    print("Hello")

print("E")
greet()

# decorator() executes and when wrapper() executes.

A
D
E
B
Hello
C


In [70]:
def decorator(func):
    
    def wrapper():
        print("Before")
        func()
        print("After")
    
    return wrapper

@decorator
def greet(name):
    print("Hello",name)

greet("Srikanth")

TypeError: decorator.<locals>.wrapper() takes 0 positional arguments but 1 was given

In [ ]:
# Wrapper fucntion doesn't take posstional arguments we can fix by *args  **kwargs (Keyword Arguments)

In [77]:
def decorator(func):
    def wrapper(*args, **kwargs):
        print("Before")
        result = func(*args, **kwargs)
        print("After")
        return result  # Wrapper function also retrun result without that retrun we get None
    return wrapper    


@decorator
def greet(name):
    return f"Hello {name}"

print(greet("Srikanth"))

Before
After
Hello Srikanth


In [ ]:
def decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@decorator
def add(a, b):
    """Add two numbers """
    return a + b

print(add.__doc__)
print(add.__name__)

# Ideally it should give docstring of function and name of the fucntion name but we are getting the None andd Wrapper which are 
# generic  how to solve @functools.wraps  

None
wrapper


In [ ]:
from functools import wraps

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@decorator
def add(a, b):
    """Add two numbers """
    return a + b

print(add.__doc__)
print(add.__name__)

#  @wraps(func) copies seleccted metadata from the original function to the wrapper and sets `__wrapper__` to the 
# original function

Add two numbers 
add


In [ ]:
from functools import wraps

def decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper

@decorator
def calculate(x):
    """Calculate a value."""
    return x + 5

print(calculate.__name__)
print(calculate.__doc__)
print(calculate(10))
print(calculate.__wrapped__(10))

wrapper
None
30


AttributeError: 'function' object has no attribute '__wrapped__'

# parameterized decorator

In [ ]:
def mutiply(factor):
    print("A")
    
    def decorator(func):
        print("B")
        
        def wrapper(x):
            print("C")
            return func(x) * factor
        return wrapper
    return decorator

@mutiply(2)
def calculate(x):
    print("D")
    return x + 10

print("E")
print(calculate(5))

# A Gets printed first becasue decarator executes first 
    

A
B
E
C
D
30


# Stacking Multiple Decorators

In [80]:
# Decorators applied from bottom to top 

def decorator_a(func):
    def wrapper():
        print("A Before")
        func()
        print("A After")
    return wrapper

def decorator_b(func):
    def wrapper():
        print("B Before")
        func()
        print("B After")
    return wrapper


@decorator_a
@decorator_b
def greet():
    print("Hello")
    
greet()



A Before
B Before
Hello
B After
A After


You have a gift (greet). You wrap it in paper B first, then wrap that in paper A.

Wrapping (bottom to top): B goes on first, A goes on last. So A is the outer layer. \
Unwrapping (when you call greet()): you open the outer layer first, so A comes off first, then B, then you reach the gift.

A-wrapper prints A Before \
A-wrapper calls what it holds, which is the B-wrapper \
B-wrapper prints B Before \
B-wrapper calls what it holds, which is the real greet\
Real greet prints Hello \
Real greet finishes, so control goes back to B-wrapper, which prints B After \
B-wrapper finishes, so control goes back to A-wrapper, which prints A After

In [ ]:
from functools import wraps
def log_error(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"Error: {e}")
            raise
    return wrapper

@log_error
def divide(a, b):
    return a/b

print(divide(10,2))
print(divide(10/0))
        

5.0


ZeroDivisionError: division by zero

In [ ]:
from functools import wraps

def multiply(factor):
    def decorator(func):
        @wraps(func)
        def wrapper(x):
            return func(x) * factor
        return wrapper
    return decorator


def add(value):
    def decorator(func):
        @wraps(func)
        def wrapper(x):
            return func(x) + value
        return wrapper
    return decorator


@multiply(2)
@add(10)
def calculate(x):
    return x


print(calculate(5)) # All decorators would call 
print(calculate.__wrapped__(5)) # Only one decorator
print(calculate.__wrapped__.__wrapped__(5)) # Only function call happens 

30
15
5
